# Real Estate Data Warehouse: Q1 2020 to Q1 2026

ETL pipeline that loads seven Q1 releases of the Saudi Ministry of Justice real estate transactions
into a PostgreSQL star schema using PySpark.

**Before running:** create the tables with `sql/create_schema.sql` and place the CSV files in `data/`
(see the README).

## 1. Spark session and database connection

In [ ]:
import os, sys, pyspark
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession
         .builder
         .appName("DW")
         .master("local[*]")
         .config("spark.driver.extraClassPath", os.path.abspath("../jars/postgresql-42.7.4.jar"))
         .getOrCreate())

In [ ]:
from getpass import getpass

# Connection settings: override with environment variables if your setup differs
pg_url = os.environ.get("PG_URL", "jdbc:postgresql://localhost:5432/DW")
pg_props = {
    "user": os.environ.get("PG_USER", "postgres"),
    "password": os.environ.get("PG_PASSWORD") or getpass("Postgres password: "),
    "driver": "org.postgresql.Driver",
}


## 2. Extract

Six of the seven files share the same 10-column layout, so they are read together.

In [ ]:
paths_id = ["../data/deals_2020_q1.csv",
            "../data/deals_2021_q1.csv",
            "../data/deals_2022_q1.csv",
            "../data/deals_2024_q1.csv",
            "../data/deals_2025_q1.csv",
            "../data/deals_2026_q1.csv"]
raw_std = spark.read.csv(paths_id, header=True, encoding="UTF-8", inferSchema=False, quote='"')

In [1]:
raw_std.printSchema()

root
 |-- المنطقة: string (nullable = true)
 |-- المدينة: string (nullable = true)
 |-- المدينة / الحي: string (nullable = true)
 |-- الرقم المرجعي للصفقة: string (nullable = true)
 |-- تاريخ الصفقة ميلادي: string (nullable = true)
 |-- تاريخ الصفقة هجري: string (nullable = true)
 |-- تصنيف العقار: string (nullable = true)
 |-- عدد العقارات: string (nullable = true)
 |-- السعر: string (nullable = true)
 |-- المساحة: string (nullable = true)



In [2]:
raw_std.count()

402265

In [3]:
raw_std.show(5, truncate=False)

+-----------------+-------+-------------------+--------------------+-------------------+-----------------+------------+------------+-------+-------+
|المنطقة          |المدينة|المدينة / الحي     |الرقم المرجعي للصفقة|تاريخ الصفقة ميلادي|تاريخ الصفقة هجري|تصنيف العقار|عدد العقارات|السعر  |المساحة|
+-----------------+-------+-------------------+--------------------+-------------------+-----------------+------------+------------+-------+-------+
|منطقة مكة المكرمه|الطائف |الطائف/ طيبة       |9470090             |2020/01/01         |1441/05/06       |سكني        |1           |300,000|835.50 |
|منطقة مكة المكرمه|الطائف |الطائف/ الجودية    |9472004             |2020/01/01         |1441/05/06       |سكني        |1           |54,000 |900.00 |
|منطقة مكة المكرمه|الطائف |الطائف/ حي البوادي |9471841             |2020/01/01         |1441/05/06       |سكني        |1           |75,000 |677.50 |
|منطقة مكة المكرمه|الطائف |الطائف/ الكدى      |9473011             |2020/01/01         |1441/05/06       |

In [ ]:
std = raw_std.toDF("region",
             "city",
             "city_district",
             "deal_id",
             "date_greg",
             "date_hijri",
             "property_class",
             "num_properties",
             "price",
             "area").withColumn(
                 "source_file",
                 F.col("_metadata.file_name")
             )

In [4]:
std.printSchema()

root
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- city_district: string (nullable = true)
 |-- deal_id: string (nullable = true)
 |-- date_greg: string (nullable = true)
 |-- date_hijri: string (nullable = true)
 |-- property_class: string (nullable = true)
 |-- num_properties: string (nullable = true)
 |-- price: string (nullable = true)
 |-- area: string (nullable = true)
 |-- source_file: string (nullable = false)



In [5]:
std.groupBy("source_file").count().orderBy("source_file").show(6)

+-----------------+-----+
|      source_file|count|
+-----------------+-----+
|deals_2020_q1.csv|80635|
|deals_2021_q1.csv|92963|
|deals_2022_q1.csv|71252|
|deals_2024_q1.csv|59099|
|deals_2025_q1.csv|65729|
|deals_2026_q1.csv|32587|
+-----------------+-----+



The 2023 file uses a different 13-column layout (it has plan, plot and property type, but no Hijri date),
so it is read on its own and mapped onto the same columns before the union.

In [ ]:
raw_std23 = spark.read.csv(path="../data/deals_2023_q1.csv", header=True, encoding="UTF-8", inferSchema=False, quote='"')
std23 = (raw_std23.toDF("deal_id",
                       "region",
                       "city",
                       "district",
                       "plan",
                       "plot",
                       "date_greg",
                       "property_class",
                       "property_type",
                       "num_properties",
                       "price",
                       "area",
                       "price_per_m2")
                       .withColumnRenamed("district", "city_district")
                       .withColumn("date_hijri", F.lit(None).cast("string"))
                       .withColumn("source_file", F.col("_metadata.file_name"))
                       .select(std.columns))

In [6]:
staging = std.unionByName(std23)
staging.count()

446356

In [7]:
staging.groupBy("source_file").count().orderBy("source_file").show()

+-----------------+-----+
|      source_file|count|
+-----------------+-----+
|deals_2020_q1.csv|80635|
|deals_2021_q1.csv|92963|
|deals_2022_q1.csv|71252|
|deals_2023_q1.csv|44091|
|deals_2024_q1.csv|59099|
|deals_2025_q1.csv|65729|
|deals_2026_q1.csv|32587|
+-----------------+-----+



## 3. Stage the raw rows in PostgreSQL

In [ ]:
staging = staging.withColumn("load_ts", F.current_timestamp())
staging.write.jdbc(url=pg_url, table="stg_deals", mode="overwrite", properties=pg_props)

In [ ]:
df_back = spark.read.jdbc(url=pg_url, table="stg_deals", properties=pg_props)

In [8]:
df_back.show()

+-----------------+------------+--------------------+-------+----------+----------+--------------+--------------+---------+------+-----------------+--------------------+
|           region|        city|       city_district|deal_id| date_greg|date_hijri|property_class|num_properties|    price|  area|      source_file|             load_ts|
+-----------------+------------+--------------------+-------+----------+----------+--------------+--------------+---------+------+-----------------+--------------------+
|منطقة مكة المكرمه|      الطائف|       الطائف/ طيبة |9470090|2020/01/01|1441/05/06|          سكني|             1|  300,000|835.50|deals_2020_q1.csv|2026-10-16 16:41:...|
|منطقة مكة المكرمه|      الطائف|    الطائف/ الجودية |9472004|2020/01/01|1441/05/06|          سكني|             1|   54,000|900.00|deals_2020_q1.csv|2026-10-16 16:41:...|
|منطقة مكة المكرمه|      الطائف| الطائف/ حي البوادي |9471841|2020/01/01|1441/05/06|          سكني|             1|   75,000|677.50|deals_2020_q1.csv|20

## 4. Transform: clean, standardize, validate

- Parse both date formats (`yyyy/M/d` and `M/d/yyyy`)
- Strip thousands separators from price and area
- Normalize Arabic region and city names (whitespace, trailing ه to ة)
- Split `city / district` and drop the `حي` prefix
- Compute price per m²

In [ ]:
from pyspark.sql import Window

spark.conf.set("spark.sql.ansi.enabled", "false")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

def to_num(c):
    return F.regexp_replace(F.col(c), r"[,\s]", "").cast("decimal(15,2)")

def norm_name(c):
    x = F.trim(F.regexp_replace(F.col(c), r"\s+", " "))
    return F.regexp_replace(x, "ه$", "ة")

In [9]:
clean = (staging
    .withColumn("deal_date",
        F.when(F.col("date_greg").rlike(r"^\d{4}/"), F.to_date("date_greg", "yyyy/M/d"))
         .otherwise(F.to_date("date_greg", "M/d/yyyy")))
    .withColumn("price", to_num("price"))
    .withColumn("area",  to_num("area"))
    .withColumn("num_properties", F.col("num_properties").cast("int"))
    .withColumn("deal_id", F.regexp_replace("deal_id", r"\D", "").cast("long"))
    .withColumn("region", norm_name("region"))
    .withColumn("region", F.regexp_replace("region", "^المنطقة ", "منطقة "))
    .withColumn("city",   norm_name("city"))
    .withColumn("district", F.trim(F.regexp_extract("city_district", r"/(.*)$", 1)))
    .withColumn("district", F.regexp_replace("district", r"^حي\s+", ""))
    .withColumn("district", F.when(F.col("district") == "", "غير محدد").otherwise(F.col("district")))
    .withColumn("property_class", F.coalesce(F.trim("property_class"), F.lit("غير محدد")))
    .withColumn("price_per_m2", F.round(F.col("price") / F.col("area"), 2))
)
clean.select("deal_date","region","city","district","property_class","price","area","price_per_m2").show(5, truncate=False)

+----------+-----------------+------+--------+--------------+---------+------+------------+
|deal_date |region           |city  |district|property_class|price    |area  |price_per_m2|
+----------+-----------------+------+--------+--------------+---------+------+------------+
|2020-01-01|منطقة مكة المكرمة|الطائف|طيبة    |سكني          |300000.00|835.50|359.07      |
|2020-01-01|منطقة مكة المكرمة|الطائف|الجودية |سكني          |54000.00 |900.00|60.00       |
|2020-01-01|منطقة مكة المكرمة|الطائف|البوادي |سكني          |75000.00 |677.50|110.70      |
|2020-01-01|منطقة مكة المكرمة|الطائف|الكدى   |سكني          |100000.00|665.14|150.34      |
|2020-01-01|منطقة مكة المكرمة|الطائف|الكدى   |سكني          |100000.00|665.14|150.34      |
+----------+-----------------+------+--------+--------------+---------+------+------------+
only showing top 5 rows


In [10]:
total = clean.count()
dq = clean.select(
    F.lit(total).alias("total_rows"),
    F.sum(F.col("deal_date").isNull().cast("int")).alias("bad_date"),
    F.sum(F.col("price").isNull().cast("int")).alias("bad_price"),
    F.sum(F.col("area").isNull().cast("int")).alias("bad_area"),
    F.sum((F.col("price") <= 0).cast("int")).alias("price_le_0"),
    F.sum((F.col("area")  <= 0).cast("int")).alias("area_le_0"),
)
dq.show()

dupes = total - clean.dropDuplicates([c for c in clean.columns if c != "load_ts"]).count()
print("exact duplicate rows:", dupes)

valid = (clean.dropDuplicates([c for c in clean.columns if c != "load_ts"])
              .filter(F.col("deal_date").isNotNull() & (F.col("price") > 0) & (F.col("area") > 0)))
print("valid rows loaded:", valid.count(), "| rejected:", total - valid.count())

+----------+--------+---------+--------+----------+---------+
|total_rows|bad_date|bad_price|bad_area|price_le_0|area_le_0|
+----------+--------+---------+--------+----------+---------+
|    446356|       4|        4|       4|         0|        0|
+----------+--------+---------+--------+----------+---------+

exact duplicate rows: 0
valid rows loaded: 446352 | rejected: 4


## 5. Build the dimensions and the fact table

In [11]:
dim_date = (valid.select(F.col("deal_date").alias("full_date")).distinct()
    .select(F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
            "full_date",
            F.dayofmonth("full_date").alias("day"),
            F.month("full_date").alias("month"),
            F.date_format("full_date", "MMMM").alias("month_name"),
            F.quarter("full_date").alias("quarter"),
            F.year("full_date").alias("year")))

dim_location = (valid.select("region", "city", "district").distinct()
    .withColumn("location_key", F.row_number().over(Window.orderBy("region", "city", "district")))
    .select("location_key", "region", "city", "district"))

dim_property = (valid.select("property_class").distinct()
    .withColumn("property_key", F.row_number().over(Window.orderBy("property_class")))
    .select("property_key", "property_class"))

dim_source = (valid.select("source_file").distinct()
    .withColumn("source_key", F.row_number().over(Window.orderBy("source_file")))
    .withColumn("year", F.regexp_extract("source_file", r"(\d{4})", 1).cast("int"))
    .withColumn("quarter", F.regexp_extract("source_file", r"_q(\d)", 1).cast("int"))
    .select("source_key", "source_file", "year", "quarter"))

for name, d in [("dim_date", dim_date), ("dim_location", dim_location),
                ("dim_property", dim_property), ("dim_source", dim_source)]:
    print(name, d.count())

dim_date 629
dim_location 11034
dim_property 5
dim_source 7


In [12]:
fact = (valid
    .join(dim_location, ["region", "city", "district"])
    .join(dim_property, ["property_class"])
    .join(dim_source.select("source_key", "source_file"), ["source_file"])
    .withColumn("date_key", F.date_format("deal_date", "yyyyMMdd").cast("int"))
    .withColumn("deal_key", F.monotonically_increasing_id() + 1)
    .select("deal_key", "deal_id", "date_key", "location_key", "property_key", "source_key",
            "num_properties", "price", "area", "price_per_m2"))
print("fact rows:", fact.count())

fact rows: 446352


## 6. Load into PostgreSQL (dimensions first, then the fact table)

In [13]:
load_props = {**pg_props, "reWriteBatchedInserts": "true"}

for name, d in [("dim_date", dim_date), ("dim_location", dim_location),
                ("dim_property", dim_property), ("dim_source", dim_source), ("fact_deals", fact)]:
    d.write.jdbc(pg_url, name, mode="append", properties=load_props)
    print("loaded", name)

loaded dim_date
loaded dim_location
loaded dim_property
loaded dim_source
loaded fact_deals
